# Import Data

In [1]:
# Running this script on a AWS Cloud SageMaker Notebook (ml.m5.2xlarge) with a volume size of 50 GB EBS

!python --version

Python 3.10.20


In [2]:
# Load required libraries

import boto3
import pandas as pd
import numpy as np
import re

In [3]:
# Downloaded the CFPB compliants data from this website on June 16, 2026
# https://www.consumerfinance.gov/data-research/consumer-complaints/#get-the-data

In [4]:
# Import the full CSV file of complaints (from every year) from my AWS S3 bucket into a single Pandas data frame

# https://aletheia-public.s3.us-east-2.amazonaws.com/complaints_16Jun2026.csv

s3 = boto3.client('s3')
obj = s3.get_object(Bucket='aletheia-public', Key='complaints_16Jun2026.csv')
df = pd.read_csv(obj['Body'])

/tmp/ipykernel_48199/411881873.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(obj['Body'])


In [5]:
# The full dataset of complaints has 15,896,496 observations

df.shape

(15896496, 16)

In [6]:
# Below is a data dictionary of all columns in the CFPB compliants data
# https://cfpb.github.io/api/ccdb/fields.html

In [7]:
# List the raw data types of all columns of the Pandas data frame

df.dtypes

Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Complaint ID                     int64
dtype: object

In [8]:
# Convert the date columns from a string object format to a datetime format with just a date (i.e. YYYY-MM-DD)

df['Date received'] = pd.to_datetime(df['Date received'].str[:10]).dt.normalize()
df['Date sent to company'] = pd.to_datetime(df['Date sent to company'].str[:10]).dt.normalize()

In [9]:
# Look at a few obs

df.head()

,Date received,Product,Sub-product,Issue,Sub-issue,Consumer complaint narrative,Company public response,Company,State,ZIP code,Tags,Submitted via,Date sent to company,Company response to consumer,Timely response?,Complaint ID
0,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,CL Holdings LLC,CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907454
1,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907265
2,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Portfolio Recovery Associates, LLC",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907334
3,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company believes it acted appropriately as aut...,"Diversified Adjustment Service, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,Yes,6907328
4,2023-04-30,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"Adler Wallach & Associates, Inc.",CA,90061,NaN,Web,2023-04-30,Closed with explanation,No,6907322


In [10]:
# Group complaints by year received and count observations

df.groupby(df['Date received'].dt.year).size()

Date received
2011       2536
2012      72368
2013     108214
2014     152909
2015     168273
2016     191294
2017     242747
2018     257133
2019     277240
2020     444214
2021     495942
2022     800257
2023    1292053
2024    2734271
2025    5443005
2026    3214040
dtype: int64

In [11]:
# Filter complaints in just the year 2025 in order to reduce the size of the Pandas data frame

df = df[df['Date received'].dt.year == 2025]

In [12]:
# Save the filtered data frame to a CSV file and share with team

df.to_csv('complaints_in_just_2025.csv', index=False)

In [13]:
# Confirm the count of observations in year 2025

df.shape

(5443005, 16)

# Explore Data

In [14]:
# Group complaints by product category and count observations

df.groupby(df['Product']).size()

Product
Checking or savings account                                  84198
Credit card                                                  89989
Credit reporting or other personal consumer reports        4810314
Debt collection                                             283206
Debt or credit management                                     4267
Money transfer, virtual currency, or money service           84618
Mortgage                                                     24694
Payday loan, title loan, personal loan, or advance loan      13135
Prepaid card                                                  6499
Student loan                                                 21314
Vehicle loan or lease                                        20771
dtype: int64

In [15]:
# Convert the Product categorical variable into a collectino of dummy variables

df = pd.get_dummies(df, columns=['Product'], drop_first=True, dtype=int)

In [16]:
# Group complaints by issue category and count observations

df.groupby(df['Issue']).size()

Issue
Advertising                                                        99
Advertising and marketing, including promotional offers          3106
Applying for a mortgage or refinancing an existing mortgage      2596
Attempts to collect debt not owed                              115683
Can't contact lender or servicer                                  203
                                                                ...  
Vehicle was repossessed or sold the vehicle                       149
Was approved for a loan, but didn't receive money                  23
Was approved for a loan, but didn't receive the money             105
Written notification about debt                                 64048
Wrong amount charged or received                                  459
Length: 90, dtype: int64

In [17]:
# Group by the tags and count obsevations

df.groupby(df['Tags']).size()

Tags
Older American                    29468
Older American, Servicemember      8004
Servicemember                    103326
dtype: int64

In [18]:
# Collapse the tags categorical feature into two separate binary features

df['Older American'] = np.where(df['Tags'].isin(['Older American','Older American, Servicemember']), 1, 0)
df['Servicemember'] = np.where(df['Tags'].isin(['Servicemember','Older American, Servicemember']), 1, 0)

In [19]:
# Group by the company public response and count obsevations

df.groupby(df['Company public response']).size()

Company public response
Company believes complaint caused principally by actions of third party outside the control or direction of the company        555
Company believes complaint is the result of an isolated error                                                                  378
Company believes complaint relates to a discontinued policy or procedure                                                         8
Company believes complaint represents an opportunity for improvement to better serve consumers                                 596
Company believes it acted appropriately as authorized by contract or law                                                     48420
Company believes the complaint is the result of a misunderstanding                                                             581
Company believes the complaint provided an opportunity to answer consumer's questions                                         2172
Company can't verify or dispute the facts in the complaint 

In [20]:
# Group by company response to consumer and count obsevations

df.groupby(df['Company response to consumer']).size()

Company response to consumer
Closed with explanation            3194212
Closed with monetary relief          26109
Closed with non-monetary relief    2212222
In progress                            365
Untimely response                    10097
dtype: int64

In [21]:
# Collapse company response to consumer into a binary target of monetary relief or not

df['Target of Monetary Relief'] = np.where(df['Company response to consumer'].isin(['Closed with monetary relief']), 1, 0)

In [22]:
# Group by target of monetary relief and count obsevations

df.groupby(df['Target of Monetary Relief']).size()

Target of Monetary Relief
0    5416896
1      26109
dtype: int64

In [23]:
# Group by timely response to consumer complaints and count obsevations

df.groupby(df['Timely response?']).size()

Timely response?
No       24595
Yes    5418410
dtype: int64

In [24]:
# Group by how complaint was submitted via and count obsevations

df.groupby(df['Submitted via']).size()

Submitted via
Phone            16421
Postal mail       7120
Referral         12187
Web            5407277
dtype: int64

In [25]:
# Convert the Product categorical variable into a collectino of dummy variables

df = pd.get_dummies(df, columns=['Submitted via'], drop_first=True, dtype=int)

In [26]:
# Count up the number of unique companies that have been the target of consumer complaints

df['Company'].nunique()

3961

In [27]:
# Find the top 15 companies with the most consumer complaints

df['Company'].value_counts().head(15)

Company
EQUIFAX, INC.                             1680538
TRANSUNION INTERMEDIATE HOLDINGS, INC.    1601851
Experian Information Solutions Inc.       1388821
Block, Inc.                                 48091
CAPITAL ONE FINANCIAL CORPORATION           34102
CBC Companies, Inc.                         30769
Early Warning Services, LLC                 24509
JPMORGAN CHASE & CO.                        24458
Resurgent Capital Services L.P.             23741
CITIBANK, N.A.                              19791
LEXISNEXIS                                  19477
WELLS FARGO & COMPANY                       19468
BANK OF AMERICA, NATIONAL ASSOCIATION       19169
ENCORE CAPITAL GROUP INC.                   17404
NAVY FEDERAL CREDIT UNION                   17276
Name: count, dtype: int64

In [28]:
# Print consumer complaints where the complaint narrative contains a specific keyword

keyword = 'Data Science'

with pd.option_context('display.max_colwidth', None):
    filtered_column = df.loc[df['Consumer complaint narrative'].str.contains(keyword, case=False, na=False), 'Consumer complaint narrative']
    print(filtered_column)

6275448    I enrolled in a one-year Postgraduate in Data Science program with XXXX XXXX XXXX XXXX from XX/XX/XXXX to XX/XX/XXXX. Toward the end of the program, I received a phone call from a XXXX XXXX representative named XXXX, who offered me an upgrade to a XXXX XXXX from XXXX de XXXX, stating it was free to explore and that there would be no financial obligation unless I chose to proceed. \n\nXXXX sent me a link to a site called Kodakco.com, assuring me it was just to check the upgrade offer or try it temporarily. Trusting XXXX XXXX as an educational provider, I clicked the link and followed instructions but I was misled into believing this was a no-obligation trial, not a financial agreement. I now realize I XXXX have unintentionally agreed to a {$1000.00} Klarna financing under false pretenses. \n\nSoon after, I was notified by Klarna of a {$1000.00} debt in my name linked to XXXX. There was no transparent checkout process or consent to financing terms. I took immediate action cont

# Feature Engineering

In [29]:
# Print out columns in the data frame

print(df.columns.tolist())

['Date received', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID', 'Product_Credit card', 'Product_Credit reporting or other personal consumer reports', 'Product_Debt collection', 'Product_Debt or credit management', 'Product_Money transfer, virtual currency, or money service', 'Product_Mortgage', 'Product_Payday loan, title loan, personal loan, or advance loan', 'Product_Prepaid card', 'Product_Student loan', 'Product_Vehicle loan or lease', 'Older American', 'Servicemember', 'Target of Monetary Relief', 'Submitted via_Postal mail', 'Submitted via_Referral', 'Submitted via_Web']


In [30]:
# Count up the words in the consumer complaint narrative

df['complaint_word_count'] = df['Consumer complaint narrative'].str.split().str.len()

In [31]:
# Count up the X letters in the consumer complaint narrative

df['letter_X_count'] = df['Consumer complaint narrative'].str.count('X')

In [32]:
# Count up the $ dollar sign in the consumer complaint narrative

df['dollar_count'] = df['Consumer complaint narrative'].str.count('$')

In [33]:
# Count up keywords related to fraud

fraud_keywords = ['fraud', 'fraudulent', 'fake', 'forgery', 'trick', 'scam', 'swindle', 'hoax', 'scheme', 'sham']
fraud_pattern = '|'.join(fraud_keywords)

df['fraud_keyword'] = df['Consumer complaint narrative'].str.contains(fraud_pattern, case=False, na=False).astype(int)

In [34]:
# Convert NaN to 0

df = df.fillna(0)

In [39]:
# Subset just the relevant columns in the data frame

relevant_columns = ['Target of Monetary Relief',
                     'Product_Credit card', 
                     'Product_Credit reporting or other personal consumer reports', 
                     'Product_Debt collection', 
                     'Product_Debt or credit management', 
                     'Product_Money transfer, virtual currency, or money service', 
                     'Product_Mortgage', 
                     'Product_Payday loan, title loan, personal loan, or advance loan', 
                     'Product_Prepaid card', 
                     'Product_Student loan', 
                     'Product_Vehicle loan or lease', 
                     'Older American', 
                     'Servicemember', 
                     'Submitted via_Postal mail', 
                     'Submitted via_Referral', 
                     'Submitted via_Web', 
                     'complaint_word_count', 
                     'letter_X_count', 
                     'dollar_count', 
                     'fraud_keyword']

subset_df = df[relevant_columns]

In [40]:
subset_df.head(20)

,Target of Monetary Relief,Product_Credit card,Product_Credit reporting or other personal consumer reports,Product_Debt collection,Product_Debt or credit management,"Product_Money transfer, virtual currency, or money service",Product_Mortgage,"Product_Payday loan, title loan, personal loan, or advance loan",Product_Prepaid card,Product_Student loan,Product_Vehicle loan or lease,Older American,Servicemember,Submitted via_Postal mail,Submitted via_Referral,Submitted via_Web,complaint_word_count,letter_X_count,dollar_count,fraud_keyword
16,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0
17,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,173.0,20.0,1.0,0
18,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,98.0,4.0,1.0,0
19,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,292.0,24.0,1.0,0
20,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0
21,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0.0,0.0,0.0,0
22,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,26.0,8.0,1.0,0
23,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,356.0,56.0,1.0,0
24,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0.0,0.0,0.0,0
25,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,717.0,208.0,1.0,0
